# Exploring the Kaputt dataset in FiftyOne

[Kaputt](https://www.kaputt-dataset.com/) is Amazon's large-scale dataset for **visual
defect detection in retail logistics** — 238k+ images of 48k+ unique items, with ~29k
defective instances ([paper](https://arxiv.org/abs/2510.05903),
[FiftyOne tutorial](https://docs.voxel51.com/tutorials/kaputt_dataset.html)). Unlike
manufacturing benchmarks, items appear in random poses with huge intra-class variation —
state-of-the-art methods don't break ~57% AUROC here.

This is the **standalone** version: it needs only **FiftyOne installed in a Python
virtual environment** — no workspace plugin, no embedded panel. You run it in a regular
JupyterLab session and it launches the FiftyOne App in your browser with `fo.launch_app()`.
It:

1. Reads the **query** Parquet index tables from `~/datasets/kaputt`
2. Resolves absolute paths to each image, crop, and mask (robust to how your download
   expanded on disk)
3. Builds a FiftyOne dataset with defect/material fields and segmentation masks
4. Indexes it for fast filtering and explores it in the App

## Dataset structure (per the tutorial)

```
~/datasets/kaputt/
├── datasets/                      # Parquet index tables
│   ├── query-train.parquet        reference-train.parquet
│   ├── query-validation.parquet   reference-validation.parquet
│   └── query-test.parquet         reference-test.parquet
├── query-image/  data/<split>/query-data/image/<capture_id>.jpg   # full tray images
├── query-crop/   data/<split>/query-data/crop/<capture_id>.jpg    # item crops
└── query-mask/   data/<split>/query-data/mask/<capture_id>.png    # segmentation masks
```

> If your download expanded into a flat `data`, `data 2`, … layout (e.g. one folder per
> tar), don't worry — step 3 **searches** for each `capture_id` by filename rather than
> assuming the exact tree, so it adapts to either layout.

## Fields we attach to each sample

- `split` — train / validation / test
- `item_id` — unique item identifier
- `defect_severity` — `none` / `minor` / `major` (a `Classification`), derived from the
  `defect` and `major_defect` boolean columns
- `defect_types` — list of types (penetration, deformation, actuation, deconstruction,
  spillage, superficial, missing_unit) as `Classifications`
- `item_material` — cardboard, plastic (hard), book (paper), … (a `Classification`)
- `is_defective` — boolean convenience field
- `crop_path` / `mask_path` — companion image paths

## Prerequisites

- A Python 3.10+ virtual environment with **FiftyOne** installed (step 0 sets this up from
  scratch if you don't have one)
- The Kaputt dataset already downloaded and extracted under `~/datasets/kaputt`
- `pandas` + `pyarrow` to read Parquet (installed in step 1)
- *(optional, steps 7 & 9)* `torch` + `torchvision` for embeddings, `transformers` for
  FastVLM — installed on demand


## 0. One-time setup: a FiftyOne virtual environment (Terminal)

If you already have a venv with FiftyOne and a Jupyter kernel, skip to step 1. Otherwise,
run these in **Terminal** once, then open this notebook on the resulting kernel.

```bash
# Python 3.11 is a safe choice for FiftyOne (3.10-3.12 all work)
python3 -m venv ~/fiftyone-env
source ~/fiftyone-env/bin/activate

pip install --upgrade pip
pip install fiftyone jupyterlab ipykernel ipywidgets pandas pyarrow

# Register this venv as a Jupyter kernel so the notebook can select it
python -m ipykernel install --user --name fiftyone-env --display-name "Python (fiftyone-env)"

# Launch JupyterLab from the venv and open this notebook
jupyter lab
```

In JupyterLab, open this notebook and select the **"Python (fiftyone-env)"** kernel
(*Kernel -> Change Kernel...*; if it isn't listed, the `ipykernel install` line hasn't run
or JupyterLab needs a relaunch). Then run the cells below in order.

> **Why a venv?** It keeps FiftyOne, the App, and this notebook's kernel in one isolated
> environment, so `import fiftyone` and `fo.launch_app()` all use the same install.


## 1. Dependencies and dataset root

The install is in its own cell on purpose: running `%pip install` and then importing the
just-installed packages in the **same** cell makes Jupyter print "you may need to restart
the kernel" and can emit harmless `Task was destroyed`/`coroutine was never awaited`
async warnings. Run the install cell once; if pandas/pyarrow were freshly installed,
restart the kernel (Kernel -> Restart) before running the imports cell. On later runs the
install is a no-op and no restart is needed.


In [ ]:
# 1a. Install Parquet deps (run once; restart the kernel if it installs anything new)
%pip install --quiet pandas pyarrow


### 1a-bis. Install the embeddings stack up front (recommended)

Step 7 needs PyTorch and a 2D-projection backend; FiftyOne's zoo models declare these
lazily, so without them you hit one missing-package error at a time. Installing them now
avoids that. If this installs anything new, restart the kernel once and re-run from 1b.


In [ ]:
# 1a-bis. Full embeddings stack (torch + torchvision + umap-learn)
import importlib.util as _ilu

_need = [m for m in ("torch", "torchvision", "umap") if _ilu.find_spec(m) is None]
if _need:
    print("Installing:", ", ".join(_need), "...")
    %pip install --quiet torch torchvision umap-learn
    print("\n\u26a0\ufe0f  Installed", ", ".join(_need),
          "-- RESTART THE KERNEL (Kernel -> Restart), then re-run from cell 1b.")
else:
    print("\u2705 Embeddings stack already present")


In [ ]:
# 1b. Imports and dataset root
import os
from pathlib import Path

import pandas as pd
import fiftyone as fo
from fiftyone import ViewField as F

KAPUTT_ROOT = Path("~/datasets/kaputt").expanduser()
assert KAPUTT_ROOT.exists(), (
    "Did not find %s — set KAPUTT_ROOT to wherever you extracted Kaputt" % KAPUTT_ROOT
)

# The Parquet index tables live under a 'datasets' subfolder
PARQUET_DIR = next(
    (p for p in [KAPUTT_ROOT / "datasets", KAPUTT_ROOT] if (p / "query-train.parquet").exists()),
    None,
)
assert PARQUET_DIR is not None, (
    "Could not find query-*.parquet under %s. Check the extraction layout." % KAPUTT_ROOT
)
print("Kaputt root :", KAPUTT_ROOT)
print("Parquet dir :", PARQUET_DIR)
print("FiftyOne     :", fo.__version__)


## 2. Peek at a query index table

Each `query-<split>.parquet` is one row per capture. Column names vary slightly by release,
so we inspect them first and then map flexibly in step 3.


In [ ]:
# 2. Inspect the schema of one split
df_preview = pd.read_parquet(PARQUET_DIR / "query-validation.parquet")
print("Rows:", len(df_preview))
print("Columns:", list(df_preview.columns))
df_preview.head(3)


## 3. Build the FiftyOne dataset

We load the **train** and **validation** query splits (test has no labels). For speed and
to keep load times and memory reasonable, set `MAX_PER_SPLIT` to cap how many samples per
split you ingest — raise it (or set to `None`) once you've confirmed everything works.

The loader is defensive about two things that bite in practice:

- **Column names**: it probes several likely names for each field (e.g. `defect_severity`
  vs `severity`), so it survives minor release differences.
- **File layout**: it builds a filename→path index by walking the image/crop/mask trees
  once, then looks each `capture_id` up — so the flat `data N` layout from a multi-tar
  extraction works just as well as the documented nested tree.


In [ ]:
# 3a. Index files on disk by stem, SEPARATELY per modality (image/crop/mask)
#
# Kaputt stores image and crop as <id>.jpg in parallel .../image/ and .../crop/
# subfolders -- same stem, different folder. A single global stem index would let a
# crop lookup return the image file (first match wins), collapsing the two. So we
# walk once and bucket each file by the modality marker in its path. Works whether
# files live under the documented query-*/data/<split>/query-data/<modality>/ tree
# or a flat "data", "data 2", ... split-per-tar extraction.
def _index_by_modality(root):
    """Return (image_idx, crop_idx, mask_idx): stem -> absolute path, per modality."""
    image_idx, crop_idx, mask_idx = {}, {}, {}
    for dirpath, _, filenames in os.walk(root):
        low = dirpath.replace(os.sep, "/").lower()
        if "/crop" in low:
            bucket, exts = crop_idx, {".jpg", ".jpeg", ".png"}
        elif "/mask" in low:
            bucket, exts = mask_idx, {".png"}
        elif "/image" in low:
            bucket, exts = image_idx, {".jpg", ".jpeg", ".png"}
        else:
            continue  # not under a modality folder; skip
        for fn in filenames:
            stem, ext = os.path.splitext(fn)
            if ext.lower() in exts:
                bucket.setdefault(stem, os.path.join(dirpath, fn))
    return image_idx, crop_idx, mask_idx

print("Indexing files on disk by modality (one-time walk)...")
IMAGE_INDEX, CROP_INDEX, MASK_INDEX = _index_by_modality(KAPUTT_ROOT)

# Fallback: if nothing matched the modality markers (unusual layout), index everything
# by stem so at least images resolve.
if not IMAGE_INDEX:
    print("No /image folder found by marker; falling back to a global stem index")
    for dirpath, _, filenames in os.walk(KAPUTT_ROOT):
        for fn in filenames:
            stem, ext = os.path.splitext(fn)
            if ext.lower() in {".jpg", ".jpeg", ".png"}:
                IMAGE_INDEX.setdefault(stem, os.path.join(dirpath, fn))

print("Indexed -> image:", len(IMAGE_INDEX),
      "| crop:", len(CROP_INDEX), "| mask:", len(MASK_INDEX))


In [ ]:
# 3b. Flexible column access + capture-id extraction
def pick(row, *names, default=None):
    for n in names:
        if n in row and pd.notna(row[n]):
            return row[n]
    return default

def to_list(val):
    if val is None:
        return []
    if isinstance(val, (list, tuple)):
        return [str(v) for v in val if v is not None]
    # numpy arrays from parquet
    try:
        import numpy as np
        if isinstance(val, np.ndarray):
            return [str(v) for v in val.tolist()]
    except Exception:
        pass
    s = str(val).strip()
    if not s or s.lower() in ("none", "nan"):
        return []
    # comma/semicolon separated fallback
    for sep in (",", ";", "|"):
        if sep in s:
            return [t.strip() for t in s.split(sep) if t.strip()]
    return [s]

def capture_id_of(row):
    cid = pick(row, "capture_id", "image_id", "id", "query_id", "sample_id")
    if cid is not None:
        return str(cid)
    # derive from an image path column if present
    p = pick(row, "image", "image_path", "query_image", "path", "filepath")
    return Path(str(p)).stem if p else None


In [ ]:
# 3c. Ingest train + validation query splits into a FiftyOne dataset
#
# Schema in this Kaputt release (from step 2):
#   capture_id, item_material, item_identifier, defect (bool),
#   major_defect (bool), defect_types (list), query_image, query_crop, query_mask
#
# Severity is derived: defect=False -> "none"; defect=True + major_defect -> "major";
# defect=True + not major_defect -> "minor". The query_* columns hold paths relative to
# the dataset root, which we resolve against disk (falling back to the filename index
# built in 3a if the relative path does not line up with your extraction layout).
DATASET_NAME = "kaputt"
MAX_PER_SPLIT = 10000  # cap per split; set to None to load everything (~95k train)
SPLITS = ["train", "validation"]

def resolve_path(rel, index, stem):
    """Resolve a parquet path column to an absolute file on disk.

    Tries the parquet-relative path first (handles the documented layout), then
    falls back to the modality-specific stem index (handles the flat data/data N
    extraction where the parquet prefix doesn't match disk). The per-modality index
    guarantees a crop lookup returns a crop file, not the same-stem image."""
    if rel is not None and pd.notna(rel):
        rel = str(rel)
        for cand in (Path(rel), KAPUTT_ROOT / rel, KAPUTT_ROOT / rel.lstrip("/")):
            if cand.exists():
                return str(cand)
    return index.get(stem)

def severity_of(row):
    is_def = bool(pick(row, "defect", default=False))
    if not is_def:
        return "none", False
    is_major = bool(pick(row, "major_defect", default=False))
    return ("major" if is_major else "minor"), True

if fo.dataset_exists(DATASET_NAME):
    fo.delete_dataset(DATASET_NAME)
dataset = fo.Dataset(DATASET_NAME, persistent=True)

samples = []
missing = 0
for split in SPLITS:
    df = pd.read_parquet(PARQUET_DIR / ("query-%s.parquet" % split))
    if MAX_PER_SPLIT:
        df = df.head(MAX_PER_SPLIT)
    print("Loading %s: %d rows" % (split, len(df)))

    for _, row in df.iterrows():
        cid = str(pick(row, "capture_id", default="") or "")
        if not cid:
            cid = Path(str(pick(row, "query_image", default=""))).stem

        img_path = resolve_path(pick(row, "query_image"), IMAGE_INDEX, cid)
        if not img_path:
            missing += 1
            continue

        severity, is_def = severity_of(row)
        material = pick(row, "item_material")
        dtypes = to_list(pick(row, "defect_types"))

        sample = fo.Sample(filepath=img_path)
        sample["split"] = split
        sample["item_id"] = str(pick(row, "item_identifier", "item_id", default=""))
        sample["defect_severity"] = fo.Classification(label=severity)
        sample["is_defective"] = is_def
        if material is not None and pd.notna(material):
            sample["item_material"] = fo.Classification(label=str(material))
        if dtypes:
            sample["defect_types"] = fo.Classifications(
                classifications=[fo.Classification(label=d) for d in dtypes]
            )

        crop = resolve_path(pick(row, "query_crop"), CROP_INDEX, cid)
        mask = resolve_path(pick(row, "query_mask"), MASK_INDEX, cid)
        if crop and crop != img_path:
            sample["crop_path"] = crop
        if mask and mask != img_path:
            sample["mask_path"] = mask
            sample["segmentation"] = fo.Segmentation(mask_path=mask)

        samples.append(sample)

dataset.add_samples(samples)
print("\nAdded %d samples (%d rows skipped for missing image)" % (len(dataset), missing))


## 4. Index for fast filtering

Indexing the fields you filter on most makes the App's sidebar and queries snappy on a
dataset this size.


In [ ]:
# 4. Create indexes on the common filter fields
for field in ("split", "is_defective", "defect_severity.label", "item_material.label"):
    try:
        dataset.create_index(field)
    except Exception as e:
        print("index", field, "skipped:", e)
print("Indexes:", dataset.list_indexes())


## 5. Explore: distributions and views

A quick numeric read on the dataset before opening the App.


In [ ]:
# 5. Summaries the App will visualize interactively
print("Total samples      :", len(dataset))
print("By split           :", dataset.count_values("split"))
print("Defective vs not   :", dataset.count_values("is_defective"))
print("By severity        :", dataset.count_values("defect_severity.label"))
print("By material (top)  :")
mats = dataset.count_values("item_material.label")
for k, v in sorted(mats.items(), key=lambda kv: -kv[1])[:10]:
    print("   %-20s %d" % (k, v))
print("Defect types (flat):")
dts = dataset.count_values("defect_types.classifications.label")
for k, v in sorted(dts.items(), key=lambda kv: -kv[1]):
    print("   %-20s %d" % (k, v))


In [ ]:
# 5b. Save a few useful views (they appear in the App's view dropdown)
defective = dataset.match(F("is_defective") == True)  # noqa: E712
dataset.save_view("defective", defective, overwrite=True)

major = dataset.match(F("defect_severity.label") == "major")
dataset.save_view("major-defects", major, overwrite=True)

# Spillage is one of the trickier defect types — save it for inspection
spillage = dataset.match(F("defect_types.classifications.label").contains("spillage"))
dataset.save_view("spillage", spillage, overwrite=True)

print("Defective :", len(defective))
print("Major     :", len(major))
print("Spillage  :", len(spillage))
print("Saved views:", dataset.list_saved_views())


## 6. Look at the App

The next cell launches the FiftyOne App in your browser. Once it opens:

- Open the view dropdown (top-left) → **defective**, **major-defects**, or **spillage**
- In the sidebar, filter by `defect_severity`, `item_material`, or `defect_types`
- Toggle the **segmentation** field to overlay item masks on the tray images
- Click a sample to inspect the full-resolution image and its labels

The App window stays live as you keep running cells — changes you make to the dataset show
up when you refresh the grid or call `session.refresh()`.

> **Tip:** if the App is already open on another dataset, point it here with
> `session.dataset = dataset` (or pick `kaputt` from the dataset selector — it lists every
> dataset in the database).


In [ ]:
# 6. Launch the FiftyOne App in your browser
session = fo.launch_app(dataset)
print("App URL:", session.url)


## 7. Curate with embeddings (optional, heavier)

To find visually similar items and surface near-duplicates or odd poses, compute a
similarity index with the FiftyOne Brain. This uses a CLIP model that requires **PyTorch**,
downloads model weights, and runs inference — it's the slowest step, so it runs on a capped
subset first.

**PyTorch is not part of the base install**, and the 2D **visualization** uses UMAP (`umap-learn`) for the best layout — cell 7b falls back to t-SNE automatically if UMAP isn't installed. For UMAP: `pip install umap-learn` in the venv, restart, re-run 1b.

**PyTorch is not part of the base install.** Cell 7a installs `torch` + `torchvision` if
they're missing; if it installs them fresh, **restart the kernel** (Kernel -> Restart) and
re-run from cell 1b before running 7b — otherwise the kernel won't see the new package.

After 7b completes, in the App you can **select an image and click the similarity icon** to
sort the grid by visual similarity, and open the **Embeddings** panel to explore clusters.


In [ ]:
# 7a. Install PyTorch if missing (restart the kernel if this installs anything new)
try:
    import torch  # noqa: F401
    print("torch already installed:", torch.__version__)
except ImportError:
    print("Installing torch + torchvision (arm64 wheels on Apple Silicon)...")
    %pip install --quiet torch torchvision
    print("\n\u26a0\ufe0f  Installed torch. RESTART THE KERNEL (Kernel -> Restart), "
          "re-run cell 1b, then run cell 7b.")


In [ ]:
# 7b. Image similarity + 2D visualization on a subset (optional; needs torch from 7a)
import importlib.util
if importlib.util.find_spec("torch") is None:
    raise RuntimeError(
        "torch is not importable yet. Run cell 7a, restart the kernel, re-run cell 1b, "
        "then run this cell."
    )

import fiftyone.brain as fob

# Reload the dataset in case the kernel was restarted after 7a (which wipes
# variables). It is persistent=True, so it survives in the database.
try:
    dataset
except NameError:
    dataset = fo.load_dataset("kaputt")
    print("Reloaded kaputt from the database:", len(dataset), "samples")

target = dataset  # embed the FULL dataset

# 1) Similarity index (skip if it already exists from a previous run)
if "kaputt_sim" not in dataset.list_brain_runs():
    fob.compute_similarity(
        target, model="clip-vit-base32-torch", brain_key="kaputt_sim"
    )
    print("Computed similarity index 'kaputt_sim'")
else:
    print("Similarity index 'kaputt_sim' already exists -- skipping")

# 2) 2D visualization. UMAP gives the nicest layout but is an extra dependency;
#    fall back to t-SNE (ships with scikit-learn) if umap-learn isn't installed.
import importlib.util
viz_method = "umap" if importlib.util.find_spec("umap") is not None else "tsne"
if viz_method == "tsne":
    print("umap-learn not found -- using t-SNE. For UMAP: pip install umap-learn, "
          "restart the kernel, re-run cell 1b, then this cell.")

if "kaputt_viz" not in dataset.list_brain_runs():
    fob.compute_visualization(
        target, model="clip-vit-base32-torch", method=viz_method, brain_key="kaputt_viz"
    )
    print("Computed visualization 'kaputt_viz' (method=%s)" % viz_method)
else:
    print("Visualization 'kaputt_viz' already exists -- skipping")

print("\nBrain runs:", dataset.list_brain_runs())
print("Open the Embeddings panel in the App (+ menu) to explore clusters.")


## 8. Build a grouped dataset (image + crop + mask slices)

Each Kaputt sample really has up to three related views: the full tray **image**, the
**crop** of the item, and the segmentation **mask**. A [grouped
dataset](https://docs.voxel51.com/user_guide/groups.html) models this natively — the three
views become *slices* of one group, and the App shows them side by side and in sync rather
than as separate samples.

This step builds `kaputt_grouped` from the flat `kaputt` dataset using the `crop_path` and
`mask_path` fields we already attached, carrying the defect/material labels onto each
group's `image` slice. The segmentation mask is **also attached as a `Segmentation` overlay**
on the image and crop slices — Kaputt masks are very dark grayscale PNGs, so viewing the raw
`mask` slice on its own looks almost black; the overlay renders the item region as a colored
region on top of the tray photo, which is what you actually want to see.


In [ ]:
# 8. Construct a grouped dataset with image / crop / mask slices
GROUPED_NAME = "kaputt_grouped"

if fo.dataset_exists(GROUPED_NAME):
    fo.delete_dataset(GROUPED_NAME)

grouped_ds = fo.Dataset(name=GROUPED_NAME, persistent=True)
grouped_ds.add_group_field("group", default="image")

LABEL_FIELDS = (
    "split", "item_id", "defect_severity", "is_defective",
    "item_material", "defect_types",
)

grouped_samples = []
for sample in dataset.iter_samples(progress=True):
    group = fo.Group()
    mask_path = sample.get_field("mask_path") if sample.has_field("mask_path") else None

    # image slice (carries the labels + the mask as a colored Segmentation overlay)
    img = fo.Sample(filepath=sample.filepath, group=group.element("image"))
    for f in LABEL_FIELDS:
        if sample.has_field(f):
            val = sample.get_field(f)
            if val is not None:
                img[f] = val
    if mask_path:
        img["segmentation"] = fo.Segmentation(mask_path=mask_path)
    grouped_samples.append(img)

    # crop slice (if present) -- also overlay the mask so the item region is visible
    crop_path = sample.get_field("crop_path") if sample.has_field("crop_path") else None
    if crop_path:
        crop = fo.Sample(filepath=crop_path, group=group.element("crop"))
        if mask_path:
            crop["segmentation"] = fo.Segmentation(mask_path=mask_path)
        grouped_samples.append(crop)

    # mask slice (if present) -- the raw mask image, for completeness in the carousel
    if mask_path:
        grouped_samples.append(
            fo.Sample(filepath=mask_path, group=group.element("mask"))
        )

grouped_ds.add_samples(grouped_samples)

print("Grouped dataset:", GROUPED_NAME)
print("Groups        :", len(grouped_ds))
print("Slices        :", grouped_ds.group_slices)
print("Media per slice:", {s: grouped_ds.match(F("group.name") == s).count()
                           for s in grouped_ds.group_slices})

# Point the open App at the grouped dataset (or pick it from the selector)
try:
    session.dataset = grouped_ds
    print("\nApp now showing 'kaputt_grouped'")
except NameError:
    print("\nLaunch or refresh the App and select 'kaputt_grouped' from the dataset picker")


**👀 See it in the App:** switch the dataset selector to **kaputt_grouped**. The grid
shows one tile per group; use the **slice selector** (upper-right of the grid) to flip
between `image`, `crop`, and `mask`, and click a tile to open the group modal where all three
views appear together in a carousel. Toggle the **segmentation** field in the sidebar to
paint the mask as a colored overlay on the image and crop slices — far more legible than the
raw `mask` slice, which is a near-black grayscale PNG. The same sidebar filters
(`defect_severity`, `item_material`, `defect_types`) apply — they live on the `image` slice.


## 9. Caption every sample with FastVLM (optional, heavier)

A vision-language model adds natural-language descriptions to each sample, so you can search
and filter the dataset by **semantic content** — and quickly audit what the model
understands versus what the ground-truth defect labels say. We use Apple's **FastVLM** from
the FiftyOne Model Zoo (a community-contributed remote source).

This downloads model weights and runs inference, so it's the heaviest step — it runs on a
capped subset by default. FastVLM needs PyTorch (installed in step 1a-bis) plus
`transformers`, which the cell installs if missing. Cell 9b auto-selects the fastest device available -- **mps (Apple-Silicon GPU)**, CUDA, or CPU -- so captioning is far faster on a Mac than the CPU fallback.


In [ ]:
# 9a. Register + install FastVLM (run once; restart kernel if it installs anything new)
import importlib.util as _ilu

if _ilu.find_spec("transformers") is None or _ilu.find_spec("timm") is None:
    print("Installing transformers + timm ...")
    %pip install --quiet transformers timm
    print("\\n\\u26a0\\ufe0f  Installed transformers -- if this is the first install, "
          "restart the kernel (Kernel -> Restart), re-run cell 1b, then continue.")
else:
    print("transformers + timm already present")

import fiftyone.zoo as foz

# Register the community FastVLM source (idempotent) and download a small variant
foz.register_zoo_model_source(
    "https://github.com/harpreetsahota204/fast_vlm", overwrite=True
)
FASTVLM_MODEL = "apple/FastVLM-0.5B"
try:
    foz.download_zoo_model(
        "https://github.com/harpreetsahota204/fast_vlm", model_name=FASTVLM_MODEL
    )
    print("FastVLM model ready:", FASTVLM_MODEL)
except Exception as e:
    print("Download note:", e)


In [ ]:
# 9b. Caption a subset and store results in a 'fastvlm_caption' field
CAPTION_N = 200   # keep small; VLM inference is slow. Raise once it works.

# Reload if the kernel was restarted after 9a
try:
    dataset
except NameError:
    dataset = fo.load_dataset("kaputt")

# Belt-and-suspenders: if any DataLoader workers do spin up, use 'fork' so the
# dynamically-imported FastVLM class doesn't need to be pickled (the default
# 'spawn'/'forkserver' methods pickle worker args and choke on it). num_workers=0
# below is the primary fix; this just makes a stray multi-worker path safe too.
import multiprocessing as _mp
try:
    _mp.set_start_method("fork", force=True)
except RuntimeError:
    pass

cap_view = dataset.limit(CAPTION_N)

# Pick the fastest available device: Apple-Silicon GPU (mps) > CUDA > CPU.
import torch
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print("Running FastVLM on device:", device)

# Pass device through to the zoo model; fall back if this build doesn't accept it
try:
    model = foz.load_zoo_model(FASTVLM_MODEL, device=device)
except Exception as e:
    print("device kwarg not accepted (%s); loading on default device" % e)
    model = foz.load_zoo_model(FASTVLM_MODEL)

model.operation = "caption"
# num_workers=0 keeps inference in the main process. FastVLM's dataset
# class is defined in a dynamically-imported module that can't be pickled
# to DataLoader worker subprocesses (PicklingError otherwise).
cap_view.apply_model(model, label_field="fastvlm_caption", num_workers=0)

# Show a few captions next to their defect labels
print("Sample captions vs. ground-truth severity:\n")
for s in cap_view.limit(5):
    cap = s.get_field("fastvlm_caption") if s.has_field("fastvlm_caption") else None
    sev = s.get_field("defect_severity") if s.has_field("defect_severity") else None
    sev = sev.label if sev is not None else "?"
    print("[%s] %s" % (sev, str(cap)[:140]))

print("\n\u2705 Captions stored in 'fastvlm_caption'. In the App, refresh and filter:")
print('   dataset.match(F("fastvlm_caption").re_match("(?i)box")) etc.')

# Refresh the open App so the new field appears, and point it back at 'kaputt'
try:
    session.dataset = dataset
    session.refresh()
except NameError:
    pass


**👀 See it in the App:** refresh the grid — each captioned sample now has a
`fastvlm_caption` field in the sidebar. Filter the dataset to captions mentioning a phrase
to audit what the model sees versus the defect labels, e.g. from the notebook:

```python
from fiftyone import ViewField as F
torn = dataset.match(F("fastvlm_caption").re_match("(?i)torn|tear|rip"))
dataset.save_view("caption-mentions-torn", torn, overwrite=True)
print(len(torn), "captioned samples mention tearing")
```


## Cleanup / notes

- The dataset is `persistent=True`, so it survives App restarts. Remove it with
  `fo.delete_dataset("kaputt")` (and `fo.delete_dataset("kaputt_grouped")`).
- Raise `MAX_PER_SPLIT` (or set it to `None`) and re-run step 3 to ingest the full splits
  once you've confirmed paths resolve correctly.
- The **reference** images (1–3 normal shots per item) can be loaded the same way from the
  `reference-*.parquet` tables if you want to build query↔reference pairs for anomaly
  detection baselines.
- The grouped dataset (step 8) and FastVLM captions (step 9) are independent add-ons — run
  either, both, or neither.

### Where to go next
- Tutorial: <https://docs.voxel51.com/tutorials/kaputt_dataset.html>
- Paper: <https://arxiv.org/abs/2510.05903>
- Grouped datasets: <https://docs.voxel51.com/user_guide/groups.html>
- Brain (similarity, uniqueness, visualization): <https://docs.voxel51.com/brain.html>
- FastVLM model: <https://docs.voxel51.com/plugins/plugins_ecosystem/fast_vlm.html>
